# 02 — Regresioni linear dhe vlerësimi i parametrave

## Konteksti dhe problemi shkencor

Ligji i Hukut dhe rënia e lirë na lejojnë të lidhim regresionin me vlerësimin e parametrave fizikë, jo vetëm me parashikimin.

### Objektivat e të nxënit

- të derivoni OLS;
- të implementoni regresionin linear;
- të analizoni rezidualet dhe pasigurinë;
- të kuptoni kondicionimin numerik.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Problemi fizik: ligji i Hukut

Përdorim $F=kx+F_0+\epsilon$. Synimi nuk është vetëm të parashikojmë $F$, por të vlerësojmë $k$ dhe të kuptojmë rezidualet.

In [ ]:
df=pd.read_csv(DATA/'hooke_measurements.csv'); x=df.zgjatja_m.to_numpy(); y=df.forca_N.to_numpy(); plt.scatter(x,y); plt.xlabel('x [m]'); plt.ylabel('F [N]'); plt.show()

## 2. Derivimi i OLS

Me $A=[\mathbf 1, x]$ minimizojmë $\|A\beta-y\|^2$. Kushti $A^T(A\beta-y)=0$ jep
$$\hat\beta=(A^TA)^{-1}A^Ty.$$
Numerikisht përdorim `solve` ose pseudoinversin, jo invers të formuar pa nevojë.

In [ ]:
A=np.c_[np.ones(len(x)),x]; beta=np.linalg.solve(A.T@A,A.T@y); beta

In [ ]:
yhat=A@beta; resid=y-yhat; print('F0=',beta[0],'k=',beta[1]); print('MSE=',np.mean(resid**2))

## 3. Kontrolli gjeometrik

Reziduali duhet të jetë ortogonal me kolonat e matricës së dizajnit.

In [ ]:
A.T@resid

## 4. Implementim nga moduli i kursit dhe krahasim me scikit-learn

In [ ]:
from src.regression.manual import ols_fit,predict_linear
from sklearn.linear_model import LinearRegression
b_manual=ols_fit(x,y); sk=LinearRegression().fit(x[:,None],y); print(b_manual, [sk.intercept_,sk.coef_[0]])

In [ ]:
xp=np.linspace(0,.105,200); plt.scatter(x,y,label='matje'); plt.plot(xp,predict_linear(xp,b_manual),label='manual'); plt.plot(xp,sk.predict(xp[:,None]),'--',label='sklearn'); plt.legend(); plt.show()

## 5. Rezidualet si instrument diagnostik

In [ ]:
plt.axhline(0,linewidth=.8); plt.scatter(x,resid); plt.xlabel('x [m]'); plt.ylabel('reziduali [N]'); plt.show()

## 6. Pasiguria e koeficientëve

Për modelin klasik, $\hat\sigma^2=RSS/(N-p)$ dhe $Cov(\hat\beta)=\hat\sigma^2(A^TA)^{-1}$.

In [ ]:
p=A.shape[1]; sigma2=(resid@resid)/(len(y)-p); cov=sigma2*np.linalg.inv(A.T@A); se=np.sqrt(np.diag(cov)); print('beta=',beta,'SE=',se)

## 7. Shembull i dytë: matja e $g$ nga rënia e lirë

Nga $s=\frac12gt^2$ mund të regresojmë $s$ ndaj $t^2$. Ky transformim vjen nga fizika, jo nga kërkim arbitrar i veçorive.

In [ ]:
rng=np.random.default_rng(12); t=np.linspace(.1,1.2,40); s=.5*9.81*t**2+rng.normal(0,.04,len(t)); m=LinearRegression(fit_intercept=True).fit((t**2)[:,None],s); print('g_hat=',2*m.coef_[0])

## 8. Rast dështimi: marrëdhënie jolineare

In [ ]:
ycurved=18*x+50*x**2+np.random.default_rng(0).normal(0,.03,len(x)); lin=LinearRegression().fit(x[:,None],ycurved); r=ycurved-lin.predict(x[:,None]); plt.scatter(x,r); plt.axhline(0); plt.show()

## 9. Ushtrime

- Implementoni regresion të ponderuar kur $\sigma_i$ rritet me $x$.
- Shtoni një matje të largët dhe krahasoni MSE me MAE.
- Krijoni dy kolona pothuajse identike dhe llogaritni `np.linalg.cond(A.T@A)`.
- Shpjegoni pse koeficienti mund të jetë i paqëndrueshëm edhe kur parashikimet janë të mira.

## 10. Shembull i plotë

Raportoni $k\pm SE_k$, interceptin, MSE dhe një grafik rezidualesh në formën e një mini-raporti laboratorik.

## 11. Playground

Ndryshoni zhurmën, numrin e matjeve dhe diapazonin e $x$. Vëzhgoni si ndryshon pasiguria e $k$.